# JSON processing

In [1]:
import json
import iosamples_3 as iosamples
from iosamples_3 import printfile
iosamples.prepare_3()

True

# Dumping

In [2]:
a = (123, 'qwerty', 4.6)
json.dumps(a)

'[123, "qwerty", 4.6]'

In [3]:
a = ('line with \n in the middle', 'кирилиця', 'кирилиця та latin')
json.dumps(a)

'["line with \\n in the middle", "\\u043a\\u0438\\u0440\\u0438\\u043b\\u0438\\u0446\\u044f", "\\u043a\\u0438\\u0440\\u0438\\u043b\\u0438\\u0446\\u044f \\u0442\\u0430 latin"]'

In [4]:
a = ('line with \n in the middle', 'кирилиця', 'кирилиця та latin')
json.dumps(a, ensure_ascii=False)

'["line with \\n in the middle", "кирилиця", "кирилиця та latin"]'

In [5]:
a = [[i+j for i in range(3)] for j in range(0,9,3)]
a

[[0, 1, 2], [3, 4, 5], [6, 7, 8]]

In [6]:
json.dumps(a)

'[[0, 1, 2], [3, 4, 5], [6, 7, 8]]'

In [7]:
print(json.dumps(a))

[[0, 1, 2], [3, 4, 5], [6, 7, 8]]


In [8]:
print(json.dumps(a, separators=(',', ':')))

[[0,1,2],[3,4,5],[6,7,8]]


In [9]:
print(json.dumps(a, indent='   '))

[
   [
      0,
      1,
      2
   ],
   [
      3,
      4,
      5
   ],
   [
      6,
      7,
      8
   ]
]


In [10]:
b = {i:i+1 for i in range(3)}
b

{0: 1, 1: 2, 2: 3}

In [11]:
print(json.dumps(b))

{"0": 1, "1": 2, "2": 3}


In [12]:
print(json.dumps(b, separators=(',', ':')))

{"0":1,"1":2,"2":3}


In [13]:
print(json.dumps(b, indent=' '))

{
 "0": 1,
 "1": 2,
 "2": 3
}


In [14]:
print(json.dumps(b, indent='--'))

{
--"0": 1,
--"1": 2,
--"2": 3
}


In [15]:
print(json.dumps(b, indent=3))

{
   "0": 1,
   "1": 2,
   "2": 3
}


## Dumping to file

In [16]:
d = {'input':{'json':'good.json', 'csv':'good.csv', 'encoding':'utf-8'},
     'output':{'fname':'res.txt', 'encoding':'utf-8'}}
fname = iosamples.JSON_SAMPLE
with open(fname, 'w', encoding='utf-8') as f:
    json.dump(d, f, indent=3)
printfile(fname)

{
   "input": {
      "json": "good.json",
      "csv": "good.csv",
      "encoding": "utf-8"
   },
   "output": {
      "fname": "res.txt",
      "encoding": "utf-8"
   }
}


# Loading

In [17]:
a = '"кирилиця\x30"'
s = json.loads(a)
print(s)
json.dumps(s)

кирилиця0


'"\\u043a\\u0438\\u0440\\u0438\\u043b\\u0438\\u0446\\u044f0"'

In [18]:
json.loads('[0, 1, 2, [3, 4, 5]]')

[0, 1, 2, [3, 4, 5]]

In [19]:
json.loads('{"0":1, "1":2, "2":3}')

{'0': 1, '1': 2, '2': 3}

In [20]:
try:
    json.loads('{0:1, "1":2, "2":3}')
except BaseException as e:
    print(type(e).__name__, ': ', e, sep='')

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


In [21]:
c = {1: ('A','B'), 2: 12.45}
print(json.dumps(c))

{"1": ["A", "B"], "2": 12.45}


In [22]:
print(json.dumps(c, indent=2))

{
  "1": [
    "A",
    "B"
  ],
  "2": 12.45
}


In [23]:
json.loads(json.dumps(c, indent=2))

{'1': ['A', 'B'], '2': 12.45}

In [24]:
fname = iosamples.JSON_SAMPLE
printfile(fname)
with open(fname, encoding='utf-8') as f:
    res = json.load(f)
res

{
   "input": {
      "json": "good.json",
      "csv": "good.csv",
      "encoding": "utf-8"
   },
   "output": {
      "fname": "res.txt",
      "encoding": "utf-8"
   }
}


{'input': {'json': 'good.json', 'csv': 'good.csv', 'encoding': 'utf-8'},
 'output': {'fname': 'res.txt', 'encoding': 'utf-8'}}

## Tricky cases (optional)

In [25]:
try:
    json.dumps(1+1j)  # комплексне число
except BaseException as e:
    print(type(e).__name__, ': ', e, sep='')

TypeError: Object of type complex is not JSON serializable


In [26]:
a = [1, 2, 3, 4]
a.append(a)
a

[1, 2, 3, 4, [...]]

In [27]:
try:
    json.dumps(a)
except BaseException as e:
    print(type(e).__name__, ': ', e, sep='')

ValueError: Circular reference detected


In [28]:
class Pair:
    def __init__(self, a, b):
        self.a = float(a)   #  не має бути відкритим, 
                            # тут відкрито тільки для спрощення прикладу
        self.b = float(b)
        
    def __repr__(self):
        return f'Pair({self.a}, {self.b})'

In [29]:
data = [12, Pair(1, 2), 'qwerty', {1:1, 2:2}]
data

[12, Pair(1.0, 2.0), 'qwerty', {1: 1, 2: 2}]

In [30]:
try:
    json.dumps(data)
except BaseException as e:
    print(type(e).__name__, ': ', e, sep='')

TypeError: Object of type Pair is not JSON serializable


Проблему можна вирішити так:

In [31]:
def f(obj):
    if isinstance(obj, Pair):
        return {'__Pair__': True, 'a':obj.a, 'b':obj.b}
    raise TypeError(f'Do not know how serialize {type(obj)}')

In [32]:
s = json.dumps(data, default=f)
s

'[12, {"__Pair__": true, "a": 1.0, "b": 2.0}, "qwerty", {"1": 1, "2": 2}]'

Але краще так:

In [33]:
class PairEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Pair):
            return {'__Pair__': True, 'a':obj.a, 'b':obj.b}
        return super().default(obj)  

In [34]:
s = json.dumps(data, cls=PairEncoder)
s

'[12, {"__Pair__": true, "a": 1.0, "b": 2.0}, "qwerty", {"1": 1, "2": 2}]'

In [35]:
def as_Pair(d):
    if '__Pair__' in d:
        p = Pair(d['a'], d['b'])
        return p
    return d

In [36]:
json.loads(s)

[12, {'__Pair__': True, 'a': 1.0, 'b': 2.0}, 'qwerty', {'1': 1, '2': 2}]

In [37]:
json.loads(s, object_hook=as_Pair)

[12, Pair(1.0, 2.0), 'qwerty', {'1': 1, '2': 2}]